# Evo-1 Inference Speedup Experiments
> **Experiments 3 and 1** from the DiT Speed & Efficiency study
> Reference: ExperimentalOverview.md

Two zero-retraining experiments run in the recommended order:

| # | Experiment | Variable | Expected gain |
|---|---|---|---|
| 3 | Temporal action-chunk reuse | Replan interval `k` | up to 25x compute saving |
| 1 | NFE (ODE steps) reduction | `num_steps` | 2-10x DiT speedup |

## Proxy architecture for Exp 3 (no client changes)
```
Eval client -> [proxy 9041-9050] -> Proxy -> [real 9001-9020] -> Evo-1 server
```
The proxy caches action chunks and returns a step-shifted slice per query.
Eval clients need zero modification.

## Hardware requirements
- **GPU**: A100 40 GB (Colab Pro/Pro+) or TC1 V100 32 GB
- **VRAM**: 10 parallel model instances x ~2.3 GB = ~23 GB
- **Runtime**: ~10 hrs for both experiments on both benchmarks

Results are saved to Drive after every sweep config.


In [ ]:
# ---- USER CONFIGURATION --------------------------------------------
WORKSPACE     = '/content/drive/MyDrive/Research/URECA/evo1_study_speedup'
NUM_TRIALS    = 10            # parallel trials per config (needs A100)
K_VALUES      = [1, 2, 5, 10, 25]    # Exp 3 temporal reuse intervals
NFE_VALUES    = [32, 10, 5, 3, 2, 1] # Exp 1 ODE steps sweep
USE_FLASH_ATTN = True         # ~15 min compile; set False to skip

# Ports -- 9000 is restricted in Colab; use 9001+
LIBERO_BASE_PORTS    = list(range(9001, 9001 + NUM_TRIALS))
METAWORLD_BASE_PORTS = list(range(9011, 9011 + NUM_TRIALS))
PROXY_PORTS          = list(range(9041, 9041 + NUM_TRIALS))
print('Config:', K_VALUES, NFE_VALUES)


In [ ]:
# 1. Setup paths and mount Drive
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Path definitions — Drive (persistent)
RESULTS_DIR  = Path(f'{WORKSPACE}/Results')
TEMPORAL_DIR = RESULTS_DIR / 'temporal_reuse'
NFE_DIR      = RESULTS_DIR / 'nfe_sweep'
PLOTS_DIR    = RESULTS_DIR / 'plots'

# Path definitions — Local (ephemeral)
CHECKPOINTS_DIR = Path('/content/checkpoints')
LOGS_DIR        = Path('/content/logs')

# Create directories
for d in [
    RESULTS_DIR, TEMPORAL_DIR, TEMPORAL_DIR/'libero', TEMPORAL_DIR/'metaworld',
    NFE_DIR, NFE_DIR/'libero', NFE_DIR/'metaworld', PLOTS_DIR,
    CHECKPOINTS_DIR, LOGS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Clean environment
os.environ.pop('PYTHONPATH', None)
os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)

# Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Enable in Runtime > Change runtime type")

gpu_name = torch.cuda.get_device_name(0)
total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ GPU: {gpu_name} ({total_mem:.1f} GB)")
print(f"✅ Workspace: {WORKSPACE}")
print(f"✅ Results: {RESULTS_DIR}")
if total_mem < 30:
    print('WARNING: < 30 GB VRAM. Consider reducing NUM_TRIALS to 3-4.')

---
## 2. Install Dependencies (3 Conda Environments)

**Critical**: Evo-1 requires `transformers==4.57.6` (NOT 5.0.0 which causes meta tensor issues).
See `EVO1_COLAB_STANDARD.md` for the canonical dependency tree.

In [ ]:
# 2. Install dependencies (3 SEPARATE conda environments - official structure)
print('📦 Installing system dependencies...')
!apt-get update -qq
!apt-get install -y -qq git wget ffmpeg libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa > /dev/null 2>&1

print('📦 Installing Miniconda...')
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/conda > /dev/null 2>&1
!rm /tmp/miniconda.sh
os.environ['PATH'] = f"/opt/conda/bin:{os.environ['PATH']}"
!conda init bash
!conda config --set always_yes yes

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

print('\n📦 Creating 3 conda environments (official Evo-1 structure)...')
print('='*60)

# Environment 1: Evo-1 server (Python 3.10)
print('\n[1/3] evo1_server (Python 3.10) - for Evo-1 model')
!conda create -n evo1_server python=3.10 -y
!conda run -n evo1_server pip install  \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# LOG: 28 Jan 2026 transformers == 5.0.0 (latest currently) causes issue with server initialisation
# specifically, issue with meta tensors and internvit initialisation.
# Use previous version 4.57.6 for functionability

!conda run -n evo1_server pip install \
  'numpy>=1.26.4,<2.0' 'transformers==4.57.6' accelerate diffusers \
  einops timm pillow opencv-python-headless \
  websockets pyyaml huggingface_hub tqdm
print('📦 Installing flash-attn (required, ~10-15 min)...')
!conda run -n evo1_server pip install flash-attn --no-build-isolation 2>&1 | tail -5 || echo '⚠️ Flash-attn failed'
print('✅ evo1_server ready')

# Environment 2: LIBERO client (Python 3.8.13 - OFFICIAL requirement)
print('\n[2/3] libero_client (Python 3.8.13) - for LIBERO benchmark')
!conda create -n libero_client python=3.8.13 -y
!conda run -n libero_client pip install \
  'numpy>=1.20,<2.0' robosuite==1.4.1 mujoco==2.3.7 \
  imageio h5py bddl websockets huggingface_hub
!conda run -n libero_client pip install \
  torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 \
  --extra-index-url https://download.pytorch.org/whl/cu113
print('✅ libero_client ready')

# Environment 3: MetaWorld client (Python 3.10)
print('\n[3/3] metaworld_client (Python 3.10) - for MetaWorld benchmark')
!conda create -n metaworld_client python=3.10 -y
!conda run -n metaworld_client pip install \
  mujoco websockets opencv-python packaging huggingface_hub metaworld gymnasium

print('✅ metaworld_client ready')

print('\n' + '='*60)
print('✅ All 3 environments created!')
!conda run -n evo1_server python -c "import sys; print(f'  evo1_server: Python {sys.version.split()[0]}')"
!conda run -n libero_client python -c "import sys; print(f'  libero_client: Python {sys.version.split()[0]}')"
!conda run -n metaworld_client python -c "import sys; print(f'  metaworld_client: Python {sys.version.split()[0]}')"

In [ ]:
# 3. Clone and install repositories in their respective environments
import os
import yaml
from pathlib import Path

print('📦 Setting up repositories...')
print('='*60)

# ==================== Clone Evo-1 ====================
print('\n[1/3] Cloning Evo-1...')
if not Path('/content/Evo-1/.git').exists():
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print('✅ Cloned Evo-1')
else:
    print('✅ Evo-1 already cloned')

# Install Evo-1 dependencies in server environment
print('\n📦 Installing Evo-1 dependencies in evo1_server...')
evo1_requirements = Path('/content/Evo-1/Evo_1/requirements.txt')
if evo1_requirements.exists():
    !conda run -n evo1_server pip install -q -r /content/Evo-1/Evo_1/requirements.txt
    print('✅ Evo-1 dependencies installed in evo1_server')
else:
    print('⚠️ WARNING: Evo-1 requirements.txt not found')

# ==================== Clone LIBERO ====================
print('\n[2/3] Cloning LIBERO...')
if not Path('/content/LIBERO_evaluation/LIBERO/.git').exists():
    !git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO_evaluation/LIBERO
    print('✅ Cloned LIBERO')
else:
    print('✅ LIBERO already cloned')

# Install LIBERO in client environment (Python 3.8.13)
print('\n📦 Installing LIBERO in libero_client...')
libero_requirements = Path('/content/LIBERO_evaluation/LIBERO/requirements.txt')
if libero_requirements.exists():
    !conda run -n libero_client pip install -q -r /content/LIBERO_evaluation/LIBERO/requirements.txt
    !conda run -n libero_client pip install -q -e /content/LIBERO_evaluation/LIBERO
    print('✅ LIBERO installed in libero_client')
else:
    print('⚠️ WARNING: LIBERO requirements.txt not found')

# ==================== Configure LIBERO ====================
print('\n[3/3] Configuring LIBERO...')
os.makedirs(os.path.expanduser('~/.libero'), exist_ok=True)
libero_cfg = os.path.expanduser('~/.libero/config.yaml')

if not os.path.exists(libero_cfg):
    cfg = {
        'benchmark_root': '/content/LIBERO_evaluation/LIBERO/libero/libero',
        'bddl_files': '/content/LIBERO_evaluation/LIBERO/libero/libero/bddl_files',
        'init_states': '/content/LIBERO_evaluation/LIBERO/libero/libero/init_files',
        'datasets': '/content/LIBERO_evaluation/LIBERO/datasets',
        'assets': '/content/LIBERO_evaluation/LIBERO/libero/libero/assets'
    }
    with open(libero_cfg, 'w') as f:
        yaml.dump(cfg, f)
    print('✅ LIBERO config created at ~/.libero/config.yaml')
else:
    print('✅ LIBERO config already exists')

# ==================== Install MetaWorld ====================
print('\n📦 Installing MetaWorld in metaworld_client...')
!conda run -n metaworld_client pip install -q metaworld
!conda run -n metaworld_client pip install -q opencv-python
print('✅ MetaWorld and dependencies installed')

# ==================== Verification ====================
print('\n' + '='*60)
print('🔍 Verifying installations...')
print('='*60)

verification_commands = [
    ('evo1_server', 'python -c "import torch; print(f\'PyTorch: {torch.__version__}\')"'),
    ('libero_client', 'python -c "import libero; print(\'LIBERO imported successfully\')"'),
    ('metaworld_client', 'python -c "import metaworld; print(\'MetaWorld imported successfully\')"'),
]

for env_name, cmd in verification_commands:
    print(f'\n{env_name}:')
    !conda run -n {env_name} {cmd}

print('\n' + '='*60)
print('✅ All repositories installed and configured!')
print('='*60)

In [ ]:
# 4. Download checkpoints
from huggingface_hub import snapshot_download

CHECKPOINTS_DIR = Path('/content/checkpoints/')

CHECKPOINTS = {
    'libero': {'repo': 'MINT-SJTU/Evo1_LIBERO', 'dir': CHECKPOINTS_DIR / 'libero'},
    'metaworld': {'repo': 'MINT-SJTU/Evo1_MetaWorld', 'dir': CHECKPOINTS_DIR / 'metaworld'}
}

print('📥 Downloading checkpoints...')
for name, cfg in CHECKPOINTS.items():
    cfg['dir'].mkdir(parents=True, exist_ok=True)
    model_file = cfg['dir'] / 'mp_rank_00_model_states.pt'

    if model_file.exists() and model_file.stat().st_size > 1_000_000:
        print(f"✅ {name}: {model_file.stat().st_size / 1e9:.2f} GB")
    else:
        print(f"⏳ Downloading {name}...")
        snapshot_download(
            repo_id=cfg['repo'],
            local_dir=str(cfg['dir']),
            local_dir_use_symlinks=False,
        )
        print(f"✅ {name} downloaded")

print('\n✅ Checkpoints ready')

---
## Write Helper Scripts
Write the proxy server and Evo-1 servers to `/content/` once.
These are reused for all sweeps.


In [ ]:
%%writefile /content/temporal_proxy_server.py
#!/usr/bin/env python3
# Lightweight WebSocket proxy for temporal action-chunk reuse (Experiment 3).
#
# Each proxy instance handles exactly ONE evaluation trial.
# Maintains its own step counter and action-chunk cache.
#
# On step % k == 0: forwards the client request to the real Evo-1 server.
# On other steps: returns the cached chunk SHIFTED so chunk[0] is the action
# for the current step.  The eval client always uses chunk[0], so it sees the
# correct action without any modification.
#
# Usage:
#   python temporal_proxy_server.py --real-port 9001 --proxy-port 9041 --reuse-k 5
import asyncio, json, websockets, argparse

def _args():
    p = argparse.ArgumentParser()
    p.add_argument("--real-port",  type=int, required=True,
                   help="Port of the real Evo-1 server")
    p.add_argument("--proxy-port", type=int, required=True,
                   help="Port to listen on for the eval client")
    p.add_argument("--reuse-k",    type=int, default=1,
                   help="Replan interval: query real server every k steps")
    return p.parse_args()

_cfg = _args()

async def _handle_client(client_ws):
    step = 0
    action_chunk = None   # list of [24-float] lists, length == horizon (50)
    chunk_idx = 0
    try:
        async with websockets.connect(
            f"ws://localhost:{_cfg.real_port}",
            max_size=100_000_000, ping_interval=120, ping_timeout=300, open_timeout=60,
        ) as real_ws:
            print(f"[Proxy:{_cfg.proxy_port}] client connected, k={_cfg.reuse_k}")
            async for message in client_ws:
                if step % _cfg.reuse_k == 0 or action_chunk is None:
                    await real_ws.send(message)
                    action_chunk = json.loads(await real_ws.recv())  # [[24f] x 50]
                    chunk_idx = 0

                horizon = len(action_chunk)
                # Shift chunk so chunk[0] == correct action for this step
                shifted = action_chunk[chunk_idx:] + [action_chunk[-1]] * chunk_idx
                await client_ws.send(json.dumps(shifted))
                chunk_idx = min(chunk_idx + 1, horizon - 1)
                step += 1

    except websockets.exceptions.ConnectionClosed:
        print(f"[Proxy:{_cfg.proxy_port}] client disconnected after {step} steps")
    except Exception as exc:
        import traceback
        print(f"[Proxy:{_cfg.proxy_port}] error: {exc}")
        traceback.print_exc()

async def _main():
    print(f"[Proxy:{_cfg.proxy_port}] listening, real={_cfg.real_port}, k={_cfg.reuse_k}")
    async with websockets.serve(
        _handle_client, "0.0.0.0", _cfg.proxy_port,
        max_size=100_000_000, ping_interval=120, ping_timeout=300,
    ):
        await asyncio.Future()

asyncio.run(_main())


In [ ]:
%%writefile /content/evo1_baseline_server.py
#!/usr/bin/env python3
# Evo-1 WebSocket inference server (baseline -- normal state normalisation).
# Usage:
#   python evo1_baseline_server.py --checkpoint /content/checkpoints/libero --port 9001
import asyncio, json, os, sys, argparse
import numpy as np
import torch

sys.path.insert(0, '/content/Evo-1')
from Evo_1.models.EVO1 import EVO1

parser = argparse.ArgumentParser()
parser.add_argument("--checkpoint", type=str, required=True)
parser.add_argument("--port",       type=int, required=True)
parser.add_argument("--name",       type=str, default="evo1_server")
args = parser.parse_args()

class Normalizer:
    def __init__(self, stats):
        self.state_min  = torch.tensor(stats["state_min"],  dtype=torch.float32)
        self.state_max  = torch.tensor(stats["state_max"],  dtype=torch.float32)
        self.action_min = torch.tensor(stats["action_min"], dtype=torch.float32)
        self.action_max = torch.tensor(stats["action_max"], dtype=torch.float32)

    def normalize_state(self, state):
        if state.ndim == 1:
            state = state.view(1, -1)
        smin = self.state_min.to(state.device, dtype=state.dtype)
        smax = self.state_max.to(state.device, dtype=state.dtype)
        return (state - smin) / (smax - smin + 1e-8) * 2.0 - 1.0

    def denormalize_action(self, action):
        if action.ndim == 1:
            action = action.view(1, -1)
        amin = self.action_min.to(action.device, dtype=action.dtype)
        amax = self.action_max.to(action.device, dtype=action.dtype)
        return (action + 1.0) / 2.0 * (amax - amin + 1e-8) + amin

def load_model(ckpt_dir):
    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
    stats  = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
    config["finetune_vlm"]            = False
    config["finetune_action_head"]    = False
    config["num_inference_timesteps"] = 32
    model = EVO1(config).eval()
    ckpt  = torch.load(os.path.join(ckpt_dir, "mp_rank_00_model_states.pt"),
                       map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["module"], strict=True)
    return model.to("cuda"), Normalizer(stats)

model, normalizer = load_model(args.checkpoint)

def decode_image(img_list):
    arr = np.array(img_list, dtype=np.uint8)
    if arr.ndim == 3 and arr.shape[2] == 3:
        arr = arr.transpose(2, 0, 1)
    return torch.from_numpy(arr).float().to("cuda") / 255.0

def infer(data):
    images      = [decode_image(img) for img in data["image"]]
    state       = torch.tensor(data["state"], dtype=torch.float32, device="cuda")
    if state.ndim == 1:
        state = state.unsqueeze(0)
    if state.shape[1] < 24:
        pad = torch.zeros(1, 24 - state.shape[1], device="cuda")
        state = torch.cat([state, pad], dim=1)
    norm_state  = normalizer.normalize_state(state).to(torch.float32)
    image_mask  = torch.tensor(data["image_mask"],   dtype=torch.int32, device="cuda")
    action_mask = torch.tensor([data["action_mask"]], dtype=torch.int32, device="cuda")
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        action = model.run_inference(
            images=images, image_mask=image_mask,
            prompt=data["prompt"], state_input=norm_state,
            action_mask=action_mask,
        )
    action = action.reshape(1, -1, 24)
    return normalizer.denormalize_action(action[0]).cpu().numpy().tolist()

import websockets

async def handle(ws):
    print(f"[{args.name}] client connected")
    try:
        async for msg in ws:
            await ws.send(json.dumps(infer(json.loads(msg))))
    except websockets.exceptions.ConnectionClosed:
        pass
    except Exception:
        import traceback; traceback.print_exc()

async def main():
    print(f"[{args.name}] listening on port {args.port}")
    async with websockets.serve(
        handle, "0.0.0.0", args.port,
        max_size=100_000_000, ping_interval=120, ping_timeout=300,
    ):
        await asyncio.Future()

asyncio.run(main())


In [ ]:
%%writefile /content/evo1_nfe_server.py
#!/usr/bin/env python3
# Evo-1 inference server with configurable NFE (ODE steps) -- Experiment 1.
# Only difference from baseline_server: --num-steps sets
# config["num_inference_timesteps"] before model init.
# Usage:
#   python evo1_nfe_server.py --checkpoint /content/checkpoints/libero --port 9001 --num-steps 5
import asyncio, json, os, sys, argparse
import numpy as np
import torch

sys.path.insert(0, '/content/Evo-1')
from Evo_1.models.EVO1 import EVO1

parser = argparse.ArgumentParser()
parser.add_argument("--checkpoint", type=str, required=True)
parser.add_argument("--port",       type=int, required=True)
parser.add_argument("--num-steps",  type=int, default=32,
                    help="Number of ODE integration steps (NFE) at inference")
parser.add_argument("--name",       type=str, default="evo1_nfe_server")
args = parser.parse_args()

class Normalizer:
    def __init__(self, stats):
        self.state_min  = torch.tensor(stats["state_min"],  dtype=torch.float32)
        self.state_max  = torch.tensor(stats["state_max"],  dtype=torch.float32)
        self.action_min = torch.tensor(stats["action_min"], dtype=torch.float32)
        self.action_max = torch.tensor(stats["action_max"], dtype=torch.float32)
    def normalize_state(self, s):
        if s.ndim == 1: s = s.view(1, -1)
        smin = self.state_min.to(s.device, dtype=s.dtype)
        smax = self.state_max.to(s.device, dtype=s.dtype)
        return (s - smin) / (smax - smin + 1e-8) * 2.0 - 1.0
    def denormalize_action(self, a):
        if a.ndim == 1: a = a.view(1, -1)
        amin = self.action_min.to(a.device, dtype=a.dtype)
        amax = self.action_max.to(a.device, dtype=a.dtype)
        return (a + 1.0) / 2.0 * (amax - amin + 1e-8) + amin

def load_model(ckpt_dir, num_steps):
    config = json.load(open(os.path.join(ckpt_dir, "config.json")))
    stats  = json.load(open(os.path.join(ckpt_dir, "norm_stats.json")))
    config["finetune_vlm"]            = False
    config["finetune_action_head"]    = False
    config["num_inference_timesteps"] = num_steps   # KEY: set before model init
    print(f"[NFE server] num_inference_timesteps = {num_steps}")
    model = EVO1(config).eval()
    ckpt  = torch.load(os.path.join(ckpt_dir, "mp_rank_00_model_states.pt"),
                       map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["module"], strict=True)
    return model.to("cuda"), Normalizer(stats)

model, normalizer = load_model(args.checkpoint, args.num_steps)

def decode_image(img_list):
    arr = np.array(img_list, dtype=np.uint8)
    if arr.ndim == 3 and arr.shape[2] == 3:
        arr = arr.transpose(2, 0, 1)
    return torch.from_numpy(arr).float().to("cuda") / 255.0

def infer(data):
    images      = [decode_image(img) for img in data["image"]]
    state       = torch.tensor(data["state"], dtype=torch.float32, device="cuda")
    if state.ndim == 1: state = state.unsqueeze(0)
    if state.shape[1] < 24:
        state = torch.cat([state, torch.zeros(1, 24 - state.shape[1], device="cuda")], 1)
    norm_state  = normalizer.normalize_state(state).to(torch.float32)
    image_mask  = torch.tensor(data["image_mask"],   dtype=torch.int32, device="cuda")
    action_mask = torch.tensor([data["action_mask"]], dtype=torch.int32, device="cuda")
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        action = model.run_inference(
            images=images, image_mask=image_mask,
            prompt=data["prompt"], state_input=norm_state,
            action_mask=action_mask,
        )
    action = action.reshape(1, -1, 24)
    return normalizer.denormalize_action(action[0]).cpu().numpy().tolist()

import websockets

async def handle(ws):
    try:
        async for msg in ws:
            await ws.send(json.dumps(infer(json.loads(msg))))
    except websockets.exceptions.ConnectionClosed:
        pass
    except Exception:
        import traceback; traceback.print_exc()

async def main():
    print(f"[{args.name}] port={args.port}, NFE={args.num_steps}")
    async with websockets.serve(
        handle, "0.0.0.0", args.port,
        max_size=100_000_000, ping_interval=120, ping_timeout=300,
    ):
        await asyncio.Future()

asyncio.run(main())


In [ ]:
import subprocess, time, json, socket
from pathlib import Path
import numpy as np

def kill_ports(ports):
    for p in ports:
        subprocess.run(f'lsof -ti:{p} | xargs kill -9 2>/dev/null || true',
                       shell=True, capture_output=True)

def wait_for_port(port, timeout_sec=120, retry_sec=5):
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        try:
            s = socket.create_connection(('localhost', port), timeout=2)
            s.close(); return True
        except OSError:
            time.sleep(retry_sec)
    print(f'  Port {port} not ready after {timeout_sec}s')
    return False

def start_servers(cmd_tpl, ports, stagger=18, wait=180):
    procs = []
    for i, port in enumerate(ports, 1):
        cmd = cmd_tpl.format(port=port, idx=i)
        procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))
        if i < len(ports): time.sleep(stagger)
    print(f'  Waiting {wait}s for servers to init...')
    time.sleep(wait)
    return procs

def kill_servers(procs, ports):
    for p in procs:
        p.terminate()
        try: p.wait(timeout=10)
        except: p.kill()
    kill_ports(ports)

def wait_clients(procs, timeout_min=90, check_sec=30):
    deadline = time.time() + timeout_min * 60
    while any(p.poll() is None for p in procs):
        if time.time() > deadline:
            for p in procs:
                if p.poll() is None: p.kill()
            break
        n = sum(1 for p in procs if p.poll() is not None)
        print(f'  {n}/{len(procs)} done', end='\r')
        time.sleep(check_sec)
    print()

def collect_sr(result_dir, label, n):
    # Parse per-trial JSON output files into a list of success-rate floats.
    rates = []
    for i in range(1, n + 1):
        f = Path(result_dir) / f'{label}_trial{i}.json'
        if not f.exists(): continue
        try:
            data = json.loads(f.read_text())
            if isinstance(data, dict):
                sr = (data.get('success_rate') or data.get('mean_success')
                      or data.get('sr'))
                if sr is None and 'results' in data:
                    sr = sum(bool(r.get('success'))
                             for r in data['results']) / len(data['results'])
            elif isinstance(data, list):
                sr = sum(bool(r.get('success')) for r in data
                         if isinstance(r, dict)) / max(1, len(data))
            else:
                sr = float(data)
            rates.append(float(sr))
        except Exception as e:
            print(f'  Could not parse {f.name}: {e}')
    return rates

print('Helpers defined.')


---
## Experiment 3: Temporal Action-Chunk Reuse

**Hypothesis:** LIBERO-Long (slow, sustained) tolerates larger `k` than
MetaWorld (fast, reactive). Differential reveals temporal-correlation structure.

**Proxy semantics:** On step `s`, the proxy returns `action_chunk[s%k:]`
padded with the last action. The client always takes `chunk[0]` -- the
correct action for step `s%k` within the replan window.

Architecture:
```
10 real servers stay alive for ALL k-values (one server per trial)
For each k: start 10 proxies -> run 10 clients -> kill proxies -> save
```


In [ ]:
# Start 10 LIBERO real servers (stay alive for all k sweeps)
print('Starting LIBERO real servers (10 x ports 9001-9010)...')
kill_ports(LIBERO_BASE_PORTS)
time.sleep(2)

libero_srv_cmd = (
    'conda run -n evo1_server python /content/evo1_baseline_server.py '
    f'--checkpoint {CHECKPOINTS_DIR}/libero '
    '--port {port} --name libero_real_{idx} '
    f'> {LOGS_DIR}/libero_real_{{port}}.log 2>&1'
)
libero_srv_procs = start_servers(libero_srv_cmd, LIBERO_BASE_PORTS)

for p in LIBERO_BASE_PORTS:
    wait_for_port(p, timeout_sec=60)
print('LIBERO servers ready.')


In [ ]:
# Exp 3A: LIBERO k-sweep
temporal_libero = {}

for k in K_VALUES:
    ckpt = TEMPORAL_DIR / 'libero' / f'k{k}_summary.json'
    if ckpt.exists():
        temporal_libero[k] = json.loads(ckpt.read_text())
        print(f'  Skip k={k} (ckpt found): SR={temporal_libero[k]["mean"]:.3f}')
        continue

    print(f'\n==== k={k} | LIBERO Temporal Reuse ====')

    # Start 10 lightweight proxies
    kill_ports(PROXY_PORTS)
    time.sleep(2)
    proxy_procs = []
    for rp, pp in zip(LIBERO_BASE_PORTS, PROXY_PORTS):
        cmd = (
            'conda run -n evo1_server python /content/temporal_proxy_server.py '
            f'--real-port {rp} --proxy-port {pp} --reuse-k {k} '
            f'> {LOGS_DIR}/proxy_{pp}_k{k}.log 2>&1'
        )
        proxy_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))
        time.sleep(0.5)
    time.sleep(5)  # proxies start fast

    # Start all eval clients in parallel
    client_procs = []
    for i, pp in enumerate(PROXY_PORTS, 1):
        out = TEMPORAL_DIR / 'libero' / f'k{k}_trial{i}.json'
        cmd = (
            'export PYTHONPATH=/content/LIBERO_evaluation/LIBERO && '
            'conda run -n libero_client python '
            '/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py '
            '--server-address localhost '
            f'--server-port {pp} '
            '--benchmark libero_90 --num-episodes 50 '
            f'--output {out} '
            f'> {LOGS_DIR}/libero_client_{pp}_k{k}.log 2>&1'
        )
        client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))

    wait_clients(client_procs, timeout_min=90)

    # Kill proxies
    for p in proxy_procs: p.terminate()
    kill_ports(PROXY_PORTS)

    # Collect and checkpoint results
    rates = collect_sr(TEMPORAL_DIR / 'libero', f'k{k}', NUM_TRIALS)
    result = {'k': k, 'rates': rates,
              'mean': float(np.mean(rates)) if rates else None,
              'std':  float(np.std(rates))  if rates else None}
    temporal_libero[k] = result
    ckpt.write_text(json.dumps(result, indent=2))
    print(f'  k={k} LIBERO: {result["mean"]:.3f} +/- {result["std"]:.3f}')

print('\nKilling LIBERO real servers...')
kill_servers(libero_srv_procs, LIBERO_BASE_PORTS)
print('LIBERO temporal sweep complete.')


---
### Experiment 3B: MetaWorld Temporal Reuse Sweep


In [ ]:
print('Starting MetaWorld real servers (10 x ports 9011-9020)...')
kill_ports(METAWORLD_BASE_PORTS)
time.sleep(2)

mw_srv_cmd = (
    'conda run -n evo1_server python /content/evo1_baseline_server.py '
    f'--checkpoint {CHECKPOINTS_DIR}/metaworld '
    '--port {port} --name mw_real_{idx} '
    f'> {LOGS_DIR}/mw_real_{{port}}.log 2>&1'
)
mw_srv_procs = start_servers(mw_srv_cmd, METAWORLD_BASE_PORTS)

for p in METAWORLD_BASE_PORTS:
    wait_for_port(p, timeout_sec=60)
print('MetaWorld servers ready.')

temporal_mw = {}

for k in K_VALUES:
    ckpt = TEMPORAL_DIR / 'metaworld' / f'k{k}_summary.json'
    if ckpt.exists():
        temporal_mw[k] = json.loads(ckpt.read_text())
        print(f'  Skip k={k} (ckpt found): SR={temporal_mw[k]["mean"]:.3f}')
        continue

    print(f'\n==== k={k} | MetaWorld Temporal Reuse ====')
    kill_ports(PROXY_PORTS)
    time.sleep(2)
    proxy_procs = []
    for rp, pp in zip(METAWORLD_BASE_PORTS, PROXY_PORTS):
        cmd = (
            'conda run -n evo1_server python /content/temporal_proxy_server.py '
            f'--real-port {rp} --proxy-port {pp} --reuse-k {k} '
            f'> {LOGS_DIR}/proxy_{pp}_k{k}_mw.log 2>&1'
        )
        proxy_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))
        time.sleep(0.5)
    time.sleep(5)

    client_procs = []
    for i, pp in enumerate(PROXY_PORTS, 1):
        out = TEMPORAL_DIR / 'metaworld' / f'k{k}_trial{i}.json'
        cmd = (
            'conda run -n metaworld_client python '
            '/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py '
            f'--server-port {pp} '
            f'--output {out} '
            f'> {LOGS_DIR}/mw_client_{pp}_k{k}.log 2>&1'
        )
        client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))

    wait_clients(client_procs, timeout_min=60)
    for p in proxy_procs: p.terminate()
    kill_ports(PROXY_PORTS)

    rates = collect_sr(TEMPORAL_DIR / 'metaworld', f'k{k}', NUM_TRIALS)
    result = {'k': k, 'rates': rates,
              'mean': float(np.mean(rates)) if rates else None,
              'std':  float(np.std(rates))  if rates else None}
    temporal_mw[k] = result
    ckpt.write_text(json.dumps(result, indent=2))
    print(f'  k={k} MetaWorld: {result["mean"]:.3f} +/- {result["std"]:.3f}')

print('\nKilling MetaWorld servers...')
kill_servers(mw_srv_procs, METAWORLD_BASE_PORTS)
print('MetaWorld temporal sweep complete.')


---
## Experiment 1: NFE (ODE Steps) Reduction

Each ODE step = 1 full DiT forward pass over all 8 layers.
```
DiT FLOPs ~ NFE x 8 layers x 2 x 50 tokens x hidden_dim^2
```
The NFE server sets `config["num_inference_timesteps"] = num_steps`
**before** model init, so the ODE solver uses the specified steps.

Note: server default is 32 steps. Paper reports 16.4 Hz on 4090d;
actual Hz on your GPU will differ.


In [ ]:
# Exp 1A: LIBERO NFE sweep
nfe_libero = {}

for nfe in NFE_VALUES:
    ckpt = NFE_DIR / 'libero' / f'nfe{nfe}_summary.json'
    if ckpt.exists():
        nfe_libero[nfe] = json.loads(ckpt.read_text())
        print(f'  Skip NFE={nfe} (ckpt found): SR={nfe_libero[nfe]["mean"]:.3f}')
        continue

    print(f'\n==== NFE={nfe} | LIBERO ====')
    kill_ports(LIBERO_BASE_PORTS)
    time.sleep(2)

    cmd_tpl = (
        'conda run -n evo1_server python /content/evo1_nfe_server.py '
        f'--checkpoint {CHECKPOINTS_DIR}/libero '
        f'--port {{port}} --num-steps {nfe} --name nfe{nfe}_{{idx}} '
        f'> {LOGS_DIR}/nfe{nfe}_libero_{{port}}.log 2>&1'
    )
    srv_procs = start_servers(cmd_tpl, LIBERO_BASE_PORTS)

    client_procs = []
    for i, port in enumerate(LIBERO_BASE_PORTS, 1):
        out = NFE_DIR / 'libero' / f'nfe{nfe}_trial{i}.json'
        cmd = (
            'export PYTHONPATH=/content/LIBERO_evaluation/LIBERO && '
            'conda run -n libero_client python '
            '/content/Evo-1/LIBERO_evaluation/libero_client_4tasks.py '
            '--server-address localhost '
            f'--server-port {port} '
            '--benchmark libero_90 --num-episodes 50 '
            f'--output {out} '
            f'> {LOGS_DIR}/nfe{nfe}_libero_client_{port}.log 2>&1'
        )
        client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))

    wait_clients(client_procs, timeout_min=90)
    kill_servers(srv_procs, LIBERO_BASE_PORTS)

    rates = collect_sr(NFE_DIR / 'libero', f'nfe{nfe}', NUM_TRIALS)
    result = {'nfe': nfe, 'rates': rates,
              'mean': float(np.mean(rates)) if rates else None,
              'std':  float(np.std(rates))  if rates else None}
    nfe_libero[nfe] = result
    ckpt.write_text(json.dumps(result, indent=2))
    print(f'  NFE={nfe} LIBERO: {result["mean"]:.3f} +/- {result["std"]:.3f}')

print('LIBERO NFE sweep complete.')


In [ ]:
# Exp 1B: MetaWorld NFE sweep
nfe_mw = {}

for nfe in NFE_VALUES:
    ckpt = NFE_DIR / 'metaworld' / f'nfe{nfe}_summary.json'
    if ckpt.exists():
        nfe_mw[nfe] = json.loads(ckpt.read_text())
        print(f'  Skip NFE={nfe} (ckpt found): SR={nfe_mw[nfe]["mean"]:.3f}')
        continue

    print(f'\n==== NFE={nfe} | MetaWorld ====')
    kill_ports(METAWORLD_BASE_PORTS)
    time.sleep(2)

    cmd_tpl = (
        'conda run -n evo1_server python /content/evo1_nfe_server.py '
        f'--checkpoint {CHECKPOINTS_DIR}/metaworld '
        f'--port {{port}} --num-steps {nfe} --name nfe{nfe}_mw_{{idx}} '
        f'> {LOGS_DIR}/nfe{nfe}_mw_{{port}}.log 2>&1'
    )
    srv_procs = start_servers(cmd_tpl, METAWORLD_BASE_PORTS)

    client_procs = []
    for i, port in enumerate(METAWORLD_BASE_PORTS, 1):
        out = NFE_DIR / 'metaworld' / f'nfe{nfe}_trial{i}.json'
        cmd = (
            'conda run -n metaworld_client python '
            '/content/Evo-1/MetaWorld_evaluation/mt50_evo1_client_prompt.py '
            f'--server-port {port} '
            f'--output {out} '
            f'> {LOGS_DIR}/nfe{nfe}_mw_client_{port}.log 2>&1'
        )
        client_procs.append(subprocess.Popen(cmd, shell=True, executable='/bin/bash'))

    wait_clients(client_procs, timeout_min=60)
    kill_servers(srv_procs, METAWORLD_BASE_PORTS)

    rates = collect_sr(NFE_DIR / 'metaworld', f'nfe{nfe}', NUM_TRIALS)
    result = {'nfe': nfe, 'rates': rates,
              'mean': float(np.mean(rates)) if rates else None,
              'std':  float(np.std(rates))  if rates else None}
    nfe_mw[nfe] = result
    ckpt.write_text(json.dumps(result, indent=2))
    print(f'  NFE={nfe} MetaWorld: {result["mean"]:.3f} +/- {result["std"]:.3f}')

print('MetaWorld NFE sweep complete.')


---
## Analysis and Visualization


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json, numpy as np
from pathlib import Path

# Reload from Drive if running after a session restart
def _load_sweep(d, key):
    out = {}
    for f in sorted(d.glob('*_summary.json')):
        dat = json.loads(f.read_text())
        out[dat[key]] = dat
    return out

if not temporal_libero:
    temporal_libero = _load_sweep(TEMPORAL_DIR / 'libero', 'k')
if not temporal_mw:
    temporal_mw = _load_sweep(TEMPORAL_DIR / 'metaworld', 'k')
if not nfe_libero:
    nfe_libero = _load_sweep(NFE_DIR / 'libero', 'nfe')
if not nfe_mw:
    nfe_mw = _load_sweep(NFE_DIR / 'metaworld', 'nfe')

def _ex(d, keys):
    # Extract (keys, means, stds) for keys with data
    ks = [k for k in keys if d.get(k) and d[k].get('mean') is not None]
    ms = [d[k]['mean'] for k in ks]
    ss = [d[k]['std']  for k in ks]
    return ks, ms, ss

# Figure 1: Temporal reuse
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dat, title, col in zip(
    axes,
    [temporal_libero, temporal_mw],
    ['LIBERO-90', 'MetaWorld MT50'],
    ['steelblue', 'darkorange'],
):
    ks, ms, ss = _ex(dat, K_VALUES)
    ax.errorbar(ks, ms, yerr=ss, marker='o', color=col,
                linewidth=2, capsize=5)
    if ms:
        ax.axhline(ms[0], linestyle='--', color='gray', alpha=0.5,
                   label=f'Baseline k=1: {ms[0]:.1%}')
    ax.set_xlabel('Replan interval k', fontsize=12)
    ax.set_ylabel('Success rate', fontsize=12)
    ax.set_title(f'Exp 3: Temporal Reuse - {title}', fontsize=13)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'temporal_reuse.pdf', bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'temporal_reuse.png', dpi=150, bbox_inches='tight')
plt.show()
print('temporal_reuse.pdf/.png saved')


In [ ]:
# Figure 2: NFE sweep
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dat, title, col in zip(
    axes,
    [nfe_libero, nfe_mw],
    ['LIBERO-90', 'MetaWorld MT50'],
    ['steelblue', 'darkorange'],
):
    nfes, ms, ss = _ex(dat, NFE_VALUES)
    ax.errorbar(nfes, ms, yerr=ss, marker='s', color=col,
                linewidth=2, capsize=5)
    if nfes:
        base_sr = dat[max(nfes)]['mean']
        ax.axhline(base_sr, linestyle='--', color='gray', alpha=0.5,
                   label=f'Baseline NFE={max(nfes)}: {base_sr:.1%}')
        for nfe, m in zip(nfes, ms):
            speedup = max(nfes) / nfe
            ax.annotate(f'{speedup:.0f}x', (nfe, m + 0.015),
                        ha='center', fontsize=8, color='gray')
    ax.set_xlabel('ODE steps (NFE)', fontsize=12)
    ax.set_ylabel('Success rate', fontsize=12)
    ax.set_title(f'Exp 1: NFE Reduction - {title}', fontsize=13)
    ax.invert_xaxis()  # high NFE = baseline on left
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'nfe_sweep.pdf', bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'nfe_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('nfe_sweep.pdf/.png saved')


In [ ]:
# Figure 3: Pareto front (combined speedup)
fig, ax = plt.subplots(figsize=(8, 6))
base_nfe = max(NFE_VALUES)
pts = []
for k in K_VALUES:
    for nfe in NFE_VALUES:
        lb = temporal_libero.get(k, {}).get('mean')
        mw = temporal_mw.get(k, {}).get('mean')
        nl = nfe_libero.get(nfe, {}).get('mean')
        nm = nfe_mw.get(nfe, {}).get('mean')
        if not all([lb, mw, nl, nm]): continue
        bl = temporal_libero.get(K_VALUES[0], {}).get('mean') or 1
        bm = temporal_mw.get(K_VALUES[0], {}).get('mean') or 1
        bnl = nfe_libero.get(base_nfe, {}).get('mean') or 1
        bnm = nfe_mw.get(base_nfe, {}).get('mean') or 1
        rel_sr = ((lb/bl * nl/bnl) + (mw/bm * nm/bnm)) / 2
        speedup = (base_nfe / nfe) * k
        pts.append((speedup, rel_sr, f'k={k},NFE={nfe}'))

if pts:
    xs, ys, lbls = zip(*pts)
    ax.scatter(xs, ys, s=60, alpha=0.7)
    for x, y, lbl in pts:
        ax.annotate(lbl, (x, y), xytext=(4, 4),
                    textcoords='offset points', fontsize=7, alpha=0.8)
    ax.axhline(0.95, linestyle='--', color='red', alpha=0.5,
               label='95% relative SR')
    ax.set_xlabel('Estimated DiT compute speedup (x baseline)', fontsize=12)
    ax.set_ylabel('Relative success rate (avg LIBERO + MetaWorld)', fontsize=12)
    ax.set_title('Pareto Front: Compute Speedup vs Performance', fontsize=13)
    ax.set_xscale('log'); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / 'pareto_front.pdf', bbox_inches='tight')
    fig.savefig(PLOTS_DIR / 'pareto_front.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Not enough data yet for Pareto plot.')


In [ ]:
# Print summary table
base_nfe = max(NFE_VALUES)
print('=' * 65)
print(f'{"Config":<22} {"LIBERO SR":>14} {"MW SR":>10} {"Speedup":>8}')
print('-' * 65)
for nfe in NFE_VALUES:
    nl = nfe_libero.get(nfe, {}).get('mean')
    nm = nfe_mw.get(nfe, {}).get('mean')
    sp = f'{base_nfe/nfe:.1f}x'
    lbl = f'NFE={nfe}'
    print(f'  {lbl:<20} {(nl or 0):.1%}         {(nm or 0):.1%}    {sp}')
print('-' * 65)
for k in K_VALUES:
    lb = temporal_libero.get(k, {}).get('mean')
    mw = temporal_mw.get(k, {}).get('mean')
    lbl = f'k={k}'
    print(f'  {lbl:<20} {(lb or 0):.1%}         {(mw or 0):.1%}    {k:.0f}x replan')
print('=' * 65)
print(f'Results at: {RESULTS_DIR}')
